<a href="https://colab.research.google.com/github/AfraazUlHaque/Non-Linear-SVM/blob/main/PENGUINS_nlsvm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# 1. Real-World Dataset Load Karna (Palmer Penguins via Seaborn)
df = sns.load_dataset('penguins').dropna()

# Binary classification ke liye 2 species select kar rahe hain: Adelie (0) aur Chinstrap (1)
df_subset = df[df['species'].isin(['Adelie', 'Chinstrap'])].copy()

# 3 Numerical Features 3D plot ke liye
X = df_subset[
    ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm']
].values
y = np.where(df_subset['species'] == 'Adelie', 0, 1)

# 2. Feature Scaling (SVM ke liye zaroori hai)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Non-Linear SVM Train Karna (RBF Kernel)
svm_model = SVC(kernel='rbf', C=3.0, gamma='scale')
svm_model.fit(X_scaled, y)

# 4. 3D Meshgrid Create Karna (Decision Boundary ke liye)
x_min, x_max = X_scaled[:, 0].min() - 0.5, X_scaled[:, 0].max() + 0.5
y_min, y_max = X_scaled[:, 1].min() - 0.5, X_scaled[:, 1].max() + 0.5
z_min, z_max = X_scaled[:, 2].min() - 0.5, X_scaled[:, 2].max() + 0.5

xx, yy, zz = np.meshgrid(
    np.linspace(x_min, x_max, 25),
    np.linspace(y_min, y_max, 25),
    np.linspace(z_min, z_max, 25),
)

mesh_points = np.c_[xx.ravel(), yy.ravel(), zz.ravel()]
Z = svm_model.decision_function(mesh_points)
Z = Z.reshape(xx.shape)

# 5. Plotly 3D Interactive Visualization
# A. Penguins Data Points Scatter Plot
scatter_points = go.Scatter3d(
    x=X_scaled[:, 0],
    y=X_scaled[:, 1],
    z=X_scaled[:, 2],
    mode='markers',
    marker=dict(
        size=6,
        color=y,
        colorscale='Tealrose',
        opacity=0.85,
        symbol='circle',
        line=dict(color='black', width=0.5),
    ),
    name='Penguins Data',
)

# B. Non-Linear SVM Decision Boundary (Isosurface)
decision_surface = go.Isosurface(
    x=xx.flatten(),
    y=yy.flatten(),
    z=zz.flatten(),
    value=Z.flatten(),
    isomin=0.0,
    isomax=0.0,
    surface=dict(count=1, fill=0.4, pattern='all'),
    colorscale='Sunset',
    opacity=0.35,
    name='SVM Decision Boundary (RBF)',
)

# Layout Setup
layout = go.Layout(
    title=(
        '<b>Palmer Penguins Dataset - Non-Linear SVM 3D Decision Boundary</b>'
    ),
    scene=dict(
        xaxis_title='Scaled Bill Length',
        yaxis_title='Scaled Bill Depth',
        zaxis_title='Scaled Flipper Length',
        bgcolor='rgb(245, 245, 245)',
    ),
    margin=dict(r=0, b=0, l=0, t=40),
)

# Figure Show Karna
fig = go.Figure(data=[scatter_points, decision_surface], layout=layout)
fig.show()